<a href="https://colab.research.google.com/github/mrunmayee3108/NeuroSolve/blob/main/member3_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 0 — Install everything first (run this cell first, then restart runtime)
!pip install -q "transformers>=4.40.0" accelerate "bitsandbytes>=0.46.1" peft sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.9 MB/s eta 0:00:00


In [ ]:
# CELL 1 — Restart reminder
# After running Cell 0, go to Runtime → Restart Runtime, then continue from Cell 2 onwards
print("Libraries installed. Please restart runtime now if this is your first run.")

Libraries installed. Please restart runtime now if this is your first run.


In [ ]:
# CELL 2 — Clear broken Phi-3 cache
import shutil, os

cache_path = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/microsoft")
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("Phi-3 cache cleared!")
else:
    print("No cache found, already clean.")

No cache found, already clean.


In [ ]:
# CELL 3 — Imports
import torch
import pandas as pd
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [ ]:
# CELL 4 — HuggingFace login
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(hf_token)
print("Authenticated successfully!")

Authenticated successfully!


In [ ]:
print("--- STEP 1: LOAD AI MODEL & ADAPTERS ---")

base_model_id = "microsoft/Phi-3.5-mini-instruct"
# Corrected adapter_path to account for nested directory after unzipping
adapter_path = "./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

--- STEP 1: LOAD AI MODEL & ADAPTERS ---


In [ ]:
# CELL 6 — Load base model (with cache bypass fix)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    revision="main"        # bypasses broken local cache, fetches latest from HuggingFace
)
print("Base model loaded!")

In [ ]:
from google.colab import files
uploaded = files.upload()  # a browser dialog will pop up — select your zip

Saving phi3-neuro-symbolic-adapter.zip to phi3-neuro-symbolic-adapter.zip


In [ ]:
!unzip -o phi3-neuro-symbolic-adapter.zip -d ./phi3-neuro-symbolic-adapter

Archive:  phi3-neuro-symbolic-adapter.zip
   creating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/adapter_config.json  
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/adapter_model.safetensors  
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/added_tokens.json  
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/README.md  
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/special_tokens_map.json  
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/tokenizer.json  
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/tokenizer.model  
  inflating: ./phi3-neuro-symbolic-adapter/phi3-neuro-symbolic-adapter/tokenizer_config.json  


In [ ]:
# CELL 7 — Load tokenizer and attach LoRA adapters
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload() # Merge LoRA layers into the base model to avoid conflicts
model.eval()
print("Tokenizer and adapters loaded!")

In [ ]:
# CELL 8 — Member 2's sandbox
print("--- STEP 2: LOAD MEMBER 2'S SANDBOX ---")

def extract_and_run_code(llm_output):
    code_match = re.search(r'```python\n(.*?)\n```', llm_output, re.DOTALL)
    if not code_match:
        return None, "Extraction Error: No python block found."
    python_code = code_match.group(1)
    local_env = {}
    try:
        exec(python_code, {}, local_env)
        if 'result' in local_env:
            return local_env['result'], None
        else:
            return None, "Execution Error: 'result' variable missing."
    except Exception as e:
        return None, f"Execution Error: {type(e).__name__}: {str(e)}"

print("Sandbox ready!")

--- STEP 2: LOAD MEMBER 2'S SANDBOX ---
Sandbox ready!


In [ ]:
# CELL 9 — Load test data
print("--- STEP 3: RUN THE AGENTIC PIPELINE ---")

# Use .head(50) for quick testing, remove .head(50) for full 1000-question run
test_df = pd.read_csv("unified_svamp_test.csv").head(50)
print(f"Loaded {len(test_df)} test questions.")

--- STEP 3: RUN THE AGENTIC PIPELINE ---
Loaded 50 test questions.


In [ ]:
# CELL 10 — Setup counters
correct_first_try = 0
correct_after_retry = 0
total_count = len(test_df)
MAX_RETRIES = 3

print(f"Testing on {total_count} questions with up to {MAX_RETRIES} retries each.")

Testing on 50 questions with up to 3 retries each.


In [ ]:
# CELL 11 — Agentic evaluation loop
for index, row in test_df.iterrows():
    question = row['question']
    ground_truth = str(row['answer']).strip()

    current_prompt = (
        f"### Instruction: Write Python code to solve the math problem. "
        f"Store the answer in 'result'.\n"
        f"### Question:\n{question}\n"
        f"### Code:\n"
    )

    is_correct = False
    final_answer = None
    final_error = None

    for attempt in range(MAX_RETRIES + 1):
        inputs = tokenizer(current_prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False  # Add this line to resolve the AttributeError
            )

        response = tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[-1]:],
            skip_special_tokens=True
        )

        calculated_answer, error = extract_and_run_code(response)

        if calculated_answer is not None:
            final_answer = calculated_answer
            try:
                if float(calculated_answer) == float(ground_truth):
                    is_correct = True
                    if attempt == 0:
                        correct_first_try += 1
                    else:
                        correct_after_retry += 1
            except ValueError:
                pass
            break

        else:
            final_error = error
            if attempt < MAX_RETRIES:
                print(f"  -> Attempt {attempt + 1} failed: {error}. Retrying...")

                current_prompt += (
                    response +
                    f"\n### Sandbox Error:\n{error}\n"
                    f"### Rewrite the code to fix the error:\n### Code:\n"
                )

    print(f"Problem {index + 1} | Ground Truth: {ground_truth} | "
          f"My Answer: {final_answer} | Correct: {is_correct} | Retries: {attempt}")

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

  -> Attempt 1 failed: Extraction Error: No python block found.. Retrying...


KeyboardInterrupt: 

In [ ]:
# CELL 12 — Final results
total_correct = correct_first_try + correct_after_retry
base_accuracy = (correct_first_try / total_count) * 100
agentic_accuracy = (total_correct / total_count) * 100

print("\n" + "="*50)
print("        NEURO-SYMBOLIC PIPELINE RESULTS")
print("="*50)
print(f"  Accuracy WITHOUT Agent (First Try) : {base_accuracy:.2f}%")
print(f"  Accuracy WITH Agent (After Retries): {agentic_accuracy:.2f}%")
print(f"  Total Gain from Symbolic Sandbox   : +{(agentic_accuracy - base_accuracy):.2f}%")
print(f"  Questions Correct                  : {total_correct}/{total_count}")
print("="*50)
